In [1]:
import os
from dotenv import load_dotenv
from FinMind.data import DataLoader
import pandas as pd

In [9]:
load_dotenv()# 會自動找目前工作目錄或上層的 .env（通常你在專案根目錄開 notebook 最順）
api = DataLoader()
api.login_by_token(api_token=os.environ["FINMIND_API_KEY"])

2026-02-08 22:37:42.728 | INFO     | FinMind.data.finmind_api:login_by_token:84 - Login success
2026-02-08 22:37:42.790 | INFO     | FinMind.data.finmind_api:login_by_token:84 - Login success


True

In [19]:
df = api.taiwan_stock_trading_daily_report(
                securities_trader_id="1440",
                date="2026-02-05",
            )


2026-02-08 23:16:26.856 | INFO     | FinMind.data.finmind_api:get_data:153 - download Dataset.TaiwanStockInfo, data_id: 
2026-02-08 23:16:27.334 | INFO     | FinMind.data.finmind_api:get_data:153 - download Dataset.TaiwanStockPrice, data_id: 
2026-02-08 23:16:30.549 | INFO     | FinMind.data.finmind_api:get_data:153 - download Dataset.TaiwanStockTradingDailyReport, data_id: 


In [20]:
df["buy_amount"] = df["buy"] * df["price"]
df["sell_amount"] = df["sell"] * df["price"]

In [21]:
agg = (
    df.groupby(["date", "stock_id"], as_index=False)
    .agg(
        securities_trader=("securities_trader", "first"),
        securities_trader_id=("securities_trader_id", "first"),
        buy=("buy", "sum"),
        sell=("sell", "sum"),
        buy_amount=("buy_amount", "sum"),
        sell_amount=("sell_amount", "sum")
    )
)
agg["net_buy"] = agg["buy"] - agg["sell"]
agg["net_buy_amount"] = agg["buy_amount"] - agg["sell_amount"]
agg["avg_price"] = agg["net_buy_amount"] / agg["net_buy"]

agg = agg.drop(["buy", "sell", "buy_amount", "sell_amount",], axis=1)

In [22]:
agg

,date,stock_id,securities_trader,securities_trader_id,net_buy,net_buy_amount,avg_price
0,2026-02-05,0050,美林,1440,2000,144050.0,72.025000
1,2026-02-05,0052,美林,1440,1000,42810.0,42.810000
2,2026-02-05,0056,美林,1440,189000,7102060.0,37.577037
3,2026-02-05,00635U,美林,1440,-6000,-319500.0,53.250000
4,2026-02-05,00646,美林,1440,4000,273800.0,68.450000
...,...,...,...,...,...,...,...
1384,2026-02-05,9943,美林,1440,-1000,-59600.0,59.600000
1385,2026-02-05,9945,美林,1440,362000,10527350.0,29.081077
1386,2026-02-05,9946,美林,1440,13000,217600.0,16.738462
1387,2026-02-05,9955,美林,1440,2000,69700.0,34.850000


In [24]:
df_new = pd.read_parquet("../data/merrill_1440_activity_2026-02-05_to_2026-02-06_20260208_223527.parquet")
df_new.loc[df_new['stock_id'] == "0056"]

,date,stock_id,securities_trader,securities_trader_id,net_buy,net_buy_amount,avg_price
2,2026-02-05,0056,美林,1440,189000,7102060.0,37.577037
1391,2026-02-06,0056,美林,1440,91000,3403170.0,37.397473


In [8]:
df_new

,date,stock_id,securities_trader,securities_trader_id,net_buy,net_buy_amount,avg_price
0,2026-02-05,0050,美林,1440,2000,144050.0,72.025000
1,2026-02-05,0052,美林,1440,1000,42810.0,42.810000
2,2026-02-05,0056,美林,1440,189000,7102060.0,37.577037
3,2026-02-05,00635U,美林,1440,-6000,-319500.0,53.250000
4,2026-02-05,00646,美林,1440,4000,273800.0,68.450000
...,...,...,...,...,...,...,...
3030,2026-02-06,9945,美林,1440,-14000,-399500.0,28.535714
3031,2026-02-06,9946,美林,1440,3620,62275.0,17.203039
3032,2026-02-06,9951,美林證券,1440,-2000,-108800.0,54.400000
3033,2026-02-06,9955,美林,1440,35909,1147547.1,31.957089
